In [65]:
import duckdb

WAREHOUSE_DB = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\warehouse\realestate_warehouse.duckdb"
PROBATE_FOLDER = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\raw\Historical Probate files"

con = duckdb.connect(WAREHOUSE_DB)

GOLD_DB = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\realestate_gold.duckdb"
con.execute(f"""
ATTACH '{GOLD_DB}' AS gold;
""")

cases_file = PROBATE_FOLDER + r"\PBTCases.txt"
parties_file = PROBATE_FOLDER + r"\PBTParties.txt"
events_file = PROBATE_FOLDER + r"\PBTEvents.txt"

In [ ]:
#see existing tables first
con.execute("SHOW TABLES").df()

,name
0,anchor_zip_key_counts
1,civil_case_defendant_addr
2,civil_case_defendant_matchprep
3,civil_case_defendant_primary
4,civil_case_defendants
...,...
88,property_match_311_2025
89,property_match_311_2026
90,tax_delinquency_2026_clean
91,tax_delinquency_events


In [ ]:
#Load PBTCases
con.execute(f"""
CREATE OR REPLACE TABLE probate_cases_raw AS
SELECT *
FROM read_csv(
    '{cases_file}',
    delim='|',
    header=True,
    all_varchar=True,
    ignore_errors=True,
    null_padding=True,
    quote='',
    escape='',
    strict_mode=false,
    max_line_size=10000000
)
""")

In [ ]:
#Load PBTParties
con.execute(f"""
CREATE OR REPLACE TABLE probate_parties_raw AS
SELECT *
FROM read_csv(
    '{parties_file}',
    delim='|',
    header=True,
    all_varchar=True,
    ignore_errors=True,
    null_padding=True,
    quote='',
    escape='',
    strict_mode=false,
    max_line_size=10000000
)
""")

In [7]:
con.execute(f"""
CREATE OR REPLACE TABLE probate_events_raw AS
SELECT *
FROM read_csv(
    '{events_file}',
    delim='|',
    header=True,
    all_varchar=True,
    ignore_errors=True,
    null_padding=True,
    quote='',
    escape='',
    strict_mode=false,
    max_line_size=10000000
)
""")

In [10]:
con.execute("SELECT COUNT(*) FROM probate_events_raw").df()

,count_star()
0,3311837


In [11]:
con.execute("SELECT * FROM probate_events_raw LIMIT 5").df()

,CaseNumber,DocumentID,FileDate,EventDesc,Comments,Number of Pages
0,472896-401,36168516,2025-12-11 00:00:00,eFile Original Petition,Application to Remove Guardian and for Appoint...,69
1,000000,14181979,2017-11-07 00:00:00,Oath,COURT INVESTIGATOR OATH,1
2,000752,20474728,1887-01-01 00:00:00,Report of Sale,None,4
3,000752,20474761,1887-01-01 00:00:00,Reports of Miscellaneous Kinds,None,12
4,000752,20474787,1887-01-01 00:00:00,Decree Approving/Confirmation of Sale of Real ...,None,3


In [12]:
#clean probate cases data
con.execute("""
CREATE OR REPLACE TABLE probate_cases_clean AS
SELECT
    TRIM(CaseNumber) AS case_number,
    TRY_CAST(TRIM(FileDate) AS DATE) AS file_date,
    TRIM("Nature Of Proceeding") AS nature_of_proceeding,
    TRIM(CourtNo) AS court_no,
    TRIM(Status) AS status
FROM probate_cases_raw
WHERE TRIM(CaseNumber) IS NOT NULL
  AND TRIM(CaseNumber) <> ''
""")

In [16]:
#clean probate parties data
con.execute("""
CREATE OR REPLACE TABLE probate_parties_clean AS
SELECT
    TRIM(CASE_NUMBER) AS case_number,
    TRIM(PARTY_ROLE_DESC) AS party_role,
    TRIM(PARTY) AS party_name,
    TRIM(PARTY_ADDR) AS party_addr,
    TRIM(PARTY_ADDR2) AS party_addr2
FROM probate_parties_raw
WHERE TRIM(CASE_NUMBER) IS NOT NULL
  AND TRIM(CASE_NUMBER) <> ''
""")

In [ ]:
#clean events
con.execute("""
CREATE OR REPLACE TABLE probate_events_clean AS
SELECT
    TRIM(CaseNumber) AS case_number,
    TRIM(DocumentID) AS document_id,
    TRY_CAST(TRIM(FileDate) AS DATE) AS file_date,
    TRIM(EventDesc) AS event_desc,
    TRIM(Comments) AS comments,
    TRY_CAST(NULLIF(TRIM("Number of Pages"), '') AS BIGINT) AS number_of_pages
FROM probate_events_raw
WHERE TRIM(CaseNumber) IS NOT NULL
  AND TRIM(CaseNumber) <> ''
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
#Inspect the exact raw schemas first
display(con.execute("DESCRIBE probate_cases_raw").df())
display(con.execute("DESCRIBE probate_parties_raw").df())
display(con.execute("DESCRIBE probate_events_raw").df())

,column_name,column_type,null,key,default,extra
0,CaseNumber,VARCHAR,YES,None,None,None
1,FileDate,VARCHAR,YES,None,None,None
2,Nature of Proceeding,VARCHAR,YES,None,None,None
3,CourtNo,VARCHAR,YES,None,None,None
4,Status,VARCHAR,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,CASE_NUMBER,VARCHAR,YES,None,None,None
1,PARTY_ROLE_DESC,VARCHAR,YES,None,None,None
2,PARTY,VARCHAR,YES,None,None,None
3,PARTY_ADDR,VARCHAR,YES,None,None,None
4,PARTY_ADDR2,VARCHAR,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,CaseNumber,VARCHAR,YES,None,None,None
1,DocumentID,VARCHAR,YES,None,None,None
2,FileDate,VARCHAR,YES,None,None,None
3,EventDesc,VARCHAR,YES,None,None,None
4,Comments,VARCHAR,YES,None,None,None
5,Number of Pages,VARCHAR,YES,None,None,None


In [19]:
con.execute("""
CREATE OR REPLACE TABLE probate_cases_clean AS
SELECT
    TRIM("CaseNumber") AS case_number,
    TRY_CAST(TRIM("FileDate") AS DATE) AS file_date,
    TRIM("Nature of Proceeding") AS nature_of_proceeding,
    TRIM("CourtNo") AS court_no,
    TRIM("Status") AS status
FROM probate_cases_raw
WHERE TRIM("CaseNumber") IS NOT NULL
  AND TRIM("CaseNumber") <> ''
""")

In [20]:
con.execute("""
CREATE OR REPLACE TABLE probate_parties_clean AS
SELECT
    TRIM(CASE_NUMBER) AS case_number,
    TRIM(PARTY_ROLE_DESC) AS party_role,
    TRIM(PARTY) AS party_name,
    TRIM(PARTY_ADDR) AS party_addr,
    TRIM(PARTY_ADDR2) AS party_addr2
FROM probate_parties_raw
WHERE TRIM(CASE_NUMBER) IS NOT NULL
  AND TRIM(CASE_NUMBER) <> ''
""")

In [21]:
con.execute("""
CREATE OR REPLACE TABLE probate_events_clean AS
SELECT
    TRIM("CaseNumber") AS case_number,
    TRIM("DocumentID") AS document_id,
    TRY_CAST(TRIM("FileDate") AS DATE) AS file_date,
    TRIM("EventDesc") AS event_desc,
    TRIM("Comments") AS comments,
    TRY_CAST(NULLIF(TRIM("Number of Pages"), '') AS BIGINT) AS number_of_pages
FROM probate_events_raw
WHERE TRIM("CaseNumber") IS NOT NULL
  AND TRIM("CaseNumber") <> ''
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
con.execute("""
SELECT 'cases' AS tbl, COUNT(*) FROM probate_cases_clean
UNION ALL
SELECT 'parties', COUNT(*) FROM probate_parties_clean
UNION ALL
SELECT 'events', COUNT(*) FROM probate_events_clean
""").df()

,tbl,count_star()
0,cases,111353
1,parties,501304
2,events,3309284


In [23]:
con.execute("""
CREATE OR REPLACE TABLE probate_case_parties_agg AS
SELECT
    case_number,

    STRING_AGG(DISTINCT party_name, ' || ') AS all_party_names,
    STRING_AGG(DISTINCT party_role, ' || ') AS all_party_roles,

    STRING_AGG(
        DISTINCT TRIM(CONCAT(COALESCE(party_addr,''), ' ', COALESCE(party_addr2,''))),
        ' || '
    ) AS all_party_addresses

FROM probate_parties_clean
GROUP BY case_number
""")

In [24]:
con.execute("""
CREATE OR REPLACE TABLE probate_case_events_agg AS
SELECT
    case_number,
    MIN(file_date) AS first_event_date,
    MAX(file_date) AS last_event_date,
    COUNT(*) AS event_count,
    STRING_AGG(DISTINCT event_desc, ' || ') AS event_types
FROM probate_events_clean
GROUP BY case_number
""")

In [25]:
#final enriched probate table
con.execute("""
CREATE OR REPLACE TABLE probate_cases_enriched AS
SELECT
    c.case_number,
    c.file_date AS case_file_date,
    c.nature_of_proceeding,
    c.status,

    p.all_party_names,
    p.all_party_roles,
    p.all_party_addresses,

    e.first_event_date,
    e.last_event_date,
    e.event_count,
    e.event_types

FROM probate_cases_clean c
LEFT JOIN probate_case_parties_agg p
    ON c.case_number = p.case_number
LEFT JOIN probate_case_events_agg e
    ON c.case_number = e.case_number
""")

In [26]:
con.execute("""
SELECT
    case_number,
    case_file_date,
    all_party_names,
    all_party_addresses
FROM probate_cases_enriched
WHERE all_party_addresses IS NOT NULL
  AND TRIM(all_party_addresses) <> ''
LIMIT 20
""").df()

,case_number,case_file_date,all_party_names,all_party_addresses
0,475576,2019-04-24,"Marshall, Sean || Lewis, Judith Ann","3525 Sage Road Unit 411 Houston, TX 77056 || 6..."
1,475580,2019-04-25,"Wallace, Claudia || Montes, Bettina || Montes,...","122 Oleander ST Lake Jackson, TX 77566 || 8118..."
2,475643,2019-04-26,"Hoerig, Beverley || Doby, Lorine M.","201 Rodriguez Rd. Yoakum, TX 77995 ||"
3,475665,2019-04-29,"Oliver, Deborah Sue || McWaters, Natalie Haggard","6607 Trailway Lane Spring, TX 77379 ||"
4,475767,2019-05-01,"McCulloch, Ann Knickerbocker || McCulloch, Wil...","|| 641 S. Ripple Creek Houston, TX 77057-1011"
5,475792,2019-05-02,"Critzos, Olga Constance || Stokes, Sandra Tate...","1611 Scenic Shore Kingwood, TX 77345 ||"
6,475810,2019-05-03,"Bostick, Aaron || Foreman, Pam || White, Isaac...","203 Beverly Dr. Lafayette, LA 70503 || 108 Bev..."
7,475836,2019-05-03,"Allen, William A. || Lopez, Ninoshkka || Lopez...","700 Louisiana, Suite 4100 Houston, TX 77002 ||..."
8,475855,2019-05-06,"Snow, Kellie Sue || Dummitt, Melva E.","|| 17622 Memorial Springs Drive Tomball, TX 7..."
9,475869,2019-05-07,"Zinn, Tara || HORN, OFELIA E","201 Caroline St. Suite 7000 Houston, TX 77002 ..."


In [ ]:
#Split aggregated probate addresses into one row per address
con.execute("""
CREATE OR REPLACE TABLE probate_case_addresses_raw AS
WITH exploded AS (
    SELECT
        case_number,
        case_file_date,
        nature_of_proceeding,
        status,
        all_party_names,
        all_party_roles,
        TRIM(UNNEST(STRING_SPLIT(all_party_addresses, ' || '))) AS raw_address
    FROM probate_cases_enriched
)
SELECT *
FROM exploded
WHERE raw_address IS NOT NULL
  AND raw_address <> ''
""")

In [ ]:
#Preview split addresses
con.execute("""
SELECT *
FROM probate_case_addresses_raw
LIMIT 20
""").df()

,case_number,case_file_date,nature_of_proceeding,status,all_party_names,all_party_roles,raw_address
0,475576,2019-04-24,Probate of Will (Independent Administration),Open,"Marshall, Sean || Lewis, Judith Ann",Deceased || Applicant || Independent Executor,"3525 Sage Road Unit 411 Houston, TX 77056"
1,475576,2019-04-24,Probate of Will (Independent Administration),Open,"Marshall, Sean || Lewis, Judith Ann",Deceased || Applicant || Independent Executor,"66 Governor Street Cumberland, RI 02864"
2,475580,2019-04-25,Small Estate,Closed,"Wallace, Claudia || Montes, Bettina || Montes,...",Applicant || Deceased,"122 Oleander ST Lake Jackson, TX 77566"
3,475580,2019-04-25,Small Estate,Closed,"Wallace, Claudia || Montes, Bettina || Montes,...",Applicant || Deceased,"8118 Azure Brook Houston, TX 77089"
4,475580,2019-04-25,Small Estate,Closed,"Wallace, Claudia || Montes, Bettina || Montes,...",Applicant || Deceased,"8702 Landwood DR Houston, TX 77040"
5,475580,2019-04-25,Small Estate,Closed,"Wallace, Claudia || Montes, Bettina || Montes,...",Applicant || Deceased,"5306 Ashmere LN Spring, TX 77379"
6,475643,2019-04-26,Probate of Will (Independent Administration),Closed,"Hoerig, Beverley || Doby, Lorine M.",Deceased || Independent Executor || Applicant,"201 Rodriguez Rd. Yoakum, TX 77995"
7,475665,2019-04-29,Probate of Will (All Other Estate Proceedings),Closed,"Oliver, Deborah Sue || McWaters, Natalie Haggard",Deceased || Applicant,"6607 Trailway Lane Spring, TX 77379"
8,475767,2019-05-01,Probate of Will (Independent Administration),Open,"McCulloch, Ann Knickerbocker || McCulloch, Wil...",Deceased || Independent Executor || Applicant,"641 S. Ripple Creek Houston, TX 77057-1011"
9,475792,2019-05-02,Probate of Will (Independent Administration),Closed,"Critzos, Olga Constance || Stokes, Sandra Tate...",Applicant || Co-Independent Executor || Deceased,"1611 Scenic Shore Kingwood, TX 77345"


In [ ]:
#Create a normalized probate address table
#This keeps both a full normalized string and a no-unit version for matching
con.execute("""
CREATE OR REPLACE TABLE probate_case_addresses_clean AS
SELECT
    case_number,
    case_file_date,
    nature_of_proceeding,
    status,
    all_party_names,
    all_party_roles,
    raw_address,

    UPPER(TRIM(raw_address)) AS addr_upper,

    -- remove punctuation that commonly breaks matching
    REGEXP_REPLACE(
        UPPER(TRIM(raw_address)),
        '[\\.,#]',
        '',
        'g'
    ) AS addr_nopunct

FROM probate_case_addresses_raw
""")

In [ ]:
#Build match keys from probate addresses
#Goal:
# - addr_key_no_unit candidate
# - addr_zip_key candidate
#
# This is a first-pass parser using regex and text cleanup.
# We strip city/state/zip suffixes as best we can and remove apartment/unit markers for no-unit matching.

con.execute("""
CREATE OR REPLACE TABLE probate_case_addresses_keyed AS
WITH base AS (
    SELECT
        *,
        -- remove trailing state/zip patterns like " TX 77056" or ", TX 77056"
        REGEXP_REPLACE(addr_nopunct, '\\s+TX\\s+\\d{5}(-\\d{4})?$', '', 'g') AS addr_no_state_zip
    FROM probate_case_addresses_clean
),
step1 AS (
    SELECT
        *,
        -- remove a trailing city if present by keeping the street portion before the final city token sequence
        REGEXP_REPLACE(addr_no_state_zip, '\\s+[A-Z ]+$', '', 'g') AS street_guess
    FROM base
),
step2 AS (
    SELECT
        *,
        -- remove common unit markers for a no-unit key
        REGEXP_REPLACE(
            street_guess,
            '\\s+(APT|UNIT|STE|SUITE|LOT|#)\\s*[A-Z0-9-]+$',
            '',
            'g'
        ) AS street_no_unit
    FROM step1
)
SELECT
    case_number,
    case_file_date,
    nature_of_proceeding,
    status,
    all_party_names,
    all_party_roles,
    raw_address,

    addr_upper,
    addr_nopunct,
    addr_no_state_zip,
    street_guess,
    street_no_unit,

    -- best first-pass no-unit key
    TRIM(REGEXP_REPLACE(street_no_unit, '\\s+', ' ', 'g')) AS probate_addr_key_no_unit,

    -- zip extracted from original cleaned address if present
    REGEXP_EXTRACT(addr_upper, '(\\d{5})(?:-\\d{4})?$', 1) AS probate_zip,

    CASE
        WHEN REGEXP_EXTRACT(addr_upper, '(\\d{5})(?:-\\d{4})?$', 1) <> ''
        THEN TRIM(REGEXP_REPLACE(street_no_unit, '\\s+', ' ', 'g')) || '|' ||
             REGEXP_EXTRACT(addr_upper, '(\\d{5})(?:-\\d{4})?$', 1)
        ELSE NULL
    END AS probate_addr_zip_key

FROM step2
""")

In [ ]:
#Preview the parsed probate keys
con.execute("""
SELECT
    case_number,
    raw_address,
    probate_addr_key_no_unit,
    probate_zip,
    probate_addr_zip_key
FROM probate_case_addresses_keyed
LIMIT 30
""").df()

,case_number,raw_address,probate_addr_key_no_unit,probate_zip,probate_addr_zip_key
0,475576,"3525 Sage Road Unit 411 Houston, TX 77056",3525 SAGE ROAD,77056,3525 SAGE ROAD|77056
1,475576,"66 Governor Street Cumberland, RI 02864",66 GOVERNOR STREET CUMBERLAND RI 02864,02864,66 GOVERNOR STREET CUMBERLAND RI 02864|02864
2,475580,"122 Oleander ST Lake Jackson, TX 77566",122,77566,122|77566
3,475580,"8118 Azure Brook Houston, TX 77089",8118,77089,8118|77089
4,475580,"8702 Landwood DR Houston, TX 77040",8702,77040,8702|77040
5,475580,"5306 Ashmere LN Spring, TX 77379",5306,77379,5306|77379
6,475643,"201 Rodriguez Rd. Yoakum, TX 77995",201,77995,201|77995
7,475665,"6607 Trailway Lane Spring, TX 77379",6607,77379,6607|77379
8,475767,"641 S. Ripple Creek Houston, TX 77057-1011",641,77057,641|77057
9,475792,"1611 Scenic Shore Kingwood, TX 77345",1611,77345,1611|77345


In [ ]:
#Match probate addresses to property_anchor
#First try addr_zip_key, then no-unit key
#
# NOTE: this assumes property_anchor already has:
# - addr_key_no_unit
# - addr_zip_key

con.execute("""
CREATE OR REPLACE TABLE probate_property_match AS
WITH zip_match AS (
    SELECT
        p.case_number,
        p.case_file_date,
        p.nature_of_proceeding,
        p.status,
        p.all_party_names,
        p.all_party_roles,
        p.raw_address,
        p.probate_addr_key_no_unit,
        p.probate_zip,
        p.probate_addr_zip_key,
        a.acct,
        a.site_addr_1,
        a.situs_full,
        'addr_zip_key' AS match_method
    FROM probate_case_addresses_keyed p
    INNER JOIN property_anchor a
        ON p.probate_addr_zip_key IS NOT NULL
       AND p.probate_addr_zip_key = a.addr_zip_key
),
no_unit_match AS (
    SELECT
        p.case_number,
        p.case_file_date,
        p.nature_of_proceeding,
        p.status,
        p.all_party_names,
        p.all_party_roles,
        p.raw_address,
        p.probate_addr_key_no_unit,
        p.probate_zip,
        p.probate_addr_zip_key,
        a.acct,
        a.site_addr_1,
        a.situs_full,
        'addr_key_no_unit' AS match_method
    FROM probate_case_addresses_keyed p
    INNER JOIN property_anchor a
        ON p.probate_addr_key_no_unit IS NOT NULL
       AND p.probate_addr_key_no_unit = a.addr_key_no_unit
)
SELECT DISTINCT *
FROM zip_match
UNION
SELECT DISTINCT *
FROM no_unit_match
""")

In [ ]:
#Check how many probate addresses matched to properties
con.execute("""
SELECT
    match_method,
    COUNT(*) AS rows_matched,
    COUNT(DISTINCT case_number) AS probate_cases_matched,
    COUNT(DISTINCT acct) AS properties_matched
FROM probate_property_match
GROUP BY 1
ORDER BY 2 DESC
""").df()

,match_method,rows_matched,probate_cases_matched,properties_matched
0,addr_key_no_unit,4989,3615,1105
1,addr_zip_key,4285,3024,1007


In [ ]:
#Preview matched probate cases
con.execute("""
SELECT
    case_number,
    case_file_date,
    raw_address,
    acct,
    site_addr_1,
    situs_full,
    match_method
FROM probate_property_match
LIMIT 50
""").df()

,case_number,case_file_date,raw_address,acct,site_addr_1,situs_full,match_method
0,519052,2023-08-29,"109 N. Post Oak Ln. Suite 300 Houston, TX 77024",1181310010001,109 N POST OAK LN,109 N POST OAK LN HOUSTON 77024,addr_zip_key
1,500804,2021-11-19,"5718 Westheimer Rd. Ste 1000 Houston, TX 77057",0681070000110,5718 WESTHEIMER RD,5718 WESTHEIMER RD HOUSTON 77057,addr_zip_key
2,503227,2022-02-16,"5444 Westheimer Suite 1260 Houston, TX 77056",0450010000293,5444 WESTHEIMER,5444 WESTHEIMER HOUSTON 77056,addr_zip_key
3,447080,2016-03-08,"5444 Westheimer Suite 1950 Houston, TX 77056",0450010000293,5444 WESTHEIMER,5444 WESTHEIMER HOUSTON 77056,addr_zip_key
4,458324,2017-05-31,"730 N Post oak Rd, Suite 100 Houston, TX 77024",0440820000012,730 N POST OAK RD,730 N POST OAK RD HOUSTON 77024,addr_zip_key
5,505288,2022-04-19,"730 N Post Oak Rd Suite 100 Houston, TX 77024",0440820000012,730 N POST OAK RD,730 N POST OAK RD HOUSTON 77024,addr_zip_key
6,443988,2015-10-30,"201 Caroline St. Suite 7000 Houston, TX 77002",0010240000001,201 CAROLINE ST,201 CAROLINE ST HOUSTON 77002,addr_zip_key
7,454850,2017-01-24,"2700 Bellefontaine St, Unit B8 Houston, TX 77025",1119180000048,2700 BELLEFONTAINE ST,2700 BELLEFONTAINE ST HOUSTON 77025,addr_zip_key
8,480978,2019-11-25,"1400 Broadfield Blvd., Ste 200 Houston, TX 77084",1147070000005,1400 BROADFIELD BLVD,1400 BROADFIELD BLVD HOUSTON 77084,addr_zip_key
9,478519,2019-08-14,"11711 Memorial Dr. Apt. 522 Houston, TX 77024",1133600000002,11711 MEMORIAL DR,11711 MEMORIAL DR HOUSTON 77024,addr_zip_key


In [ ]:
#inspect one known address if you want to test a specific probate case manually
#Replace the address fragment as needed
con.execute("""
SELECT
    case_number,
    case_file_date,
    raw_address,
    probate_addr_key_no_unit,
    probate_addr_zip_key
FROM probate_case_addresses_keyed
WHERE case_number ILIKE '519052'
""").df()

,case_number,case_file_date,raw_address,probate_addr_key_no_unit,probate_addr_zip_key
0,519052,2023-08-29,"109 N. Post Oak Ln. Suite 300 Houston, TX 77024",109 N POST OAK LN,109 N POST OAK LN|77024
1,519052,2023-08-29,"269 Hawk Hill Ln Orcas, WA 98280",269 HAWK HILL LN ORCAS WA 98280,269 HAWK HILL LN ORCAS WA 98280|98280


In [ ]:
#Keep one row per probate party/address with role retained
con.execute("""
CREATE OR REPLACE TABLE probate_party_addresses AS
SELECT
    p.case_number,
    c.file_date AS case_file_date,
    c.nature_of_proceeding,
    c.status,
    p.party_role,
    p.party_name,
    TRIM(CONCAT(COALESCE(p.party_addr, ''), ' ', COALESCE(p.party_addr2, ''))) AS raw_address
FROM probate_parties_clean p
LEFT JOIN probate_cases_clean c
    ON p.case_number = c.case_number
WHERE TRIM(CONCAT(COALESCE(p.party_addr, ''), ' ', COALESCE(p.party_addr2, ''))) <> ''
""")

In [ ]:
#Normalize address text
con.execute("""
CREATE OR REPLACE TABLE probate_party_addresses_norm AS
SELECT
    *,
    UPPER(TRIM(raw_address)) AS addr_upper,
    REGEXP_REPLACE(UPPER(TRIM(raw_address)), '[\\.,#]', '', 'g') AS addr_nopunct
FROM probate_party_addresses
""")

In [ ]:
#Filter OUT obvious non-property / business / office addresses
#You can tune this list later, but this is a strong first pass
con.execute("""
CREATE OR REPLACE TABLE probate_party_addresses_filtered AS
SELECT *
FROM probate_party_addresses_norm
WHERE raw_address IS NOT NULL
  AND TRIM(raw_address) <> ''
  AND (
        LOWER(party_role) LIKE '%deceased%'
        OR LOWER(party_role) LIKE '%applicant%'
        OR LOWER(party_role) LIKE '%executor%'
        OR LOWER(party_role) LIKE '%independent executor%'
        OR LOWER(party_role) LIKE '%administrator%'
      )
  AND addr_nopunct NOT LIKE '%SUITE%'
  AND addr_nopunct NOT LIKE '%STE %'
  AND addr_nopunct NOT LIKE '%FLOOR%'
  AND addr_nopunct NOT LIKE '%LAW%'
  AND addr_nopunct NOT LIKE '%ATTORNEY%'
  AND addr_nopunct NOT LIKE '%LLP%'
  AND addr_nopunct NOT LIKE '%PC %'
  AND addr_nopunct NOT LIKE '%BANK%'
  AND addr_nopunct NOT LIKE '%BLVD SUITE%'
  AND addr_nopunct NOT LIKE '%ROAD SUITE%'
  AND addr_nopunct NOT LIKE '%RD SUITE%'
  AND addr_nopunct NOT LIKE '%ST SUITE%'
  AND addr_nopunct NOT LIKE '%AVE SUITE%'
  AND addr_nopunct NOT LIKE '%PO BOX%'
""")

In [ ]:
#Build first-pass matching keys
con.execute("""
CREATE OR REPLACE TABLE probate_party_addresses_keyed AS
WITH base AS (
    SELECT
        *,
        REGEXP_REPLACE(addr_nopunct, '\\s+TX\\s+\\d{5}(-\\d{4})?$', '', 'g') AS addr_no_state_zip
    FROM probate_party_addresses_filtered
),
step1 AS (
    SELECT
        *,
        REGEXP_REPLACE(addr_no_state_zip, '\\s+[A-Z ]+$', '', 'g') AS street_guess
    FROM base
),
step2 AS (
    SELECT
        *,
        REGEXP_REPLACE(
            street_guess,
            '\\s+(APT|UNIT|STE|SUITE|LOT|#)\\s*[A-Z0-9-]+$',
            '',
            'g'
        ) AS street_no_unit
    FROM step1
)
SELECT
    case_number,
    case_file_date,
    nature_of_proceeding,
    status,
    party_role,
    party_name,
    raw_address,
    TRIM(REGEXP_REPLACE(street_no_unit, '\\s+', ' ', 'g')) AS probate_addr_key_no_unit,
    REGEXP_EXTRACT(addr_upper, '(\\d{5})(?:-\\d{4})?$', 1) AS probate_zip,
    CASE
        WHEN REGEXP_EXTRACT(addr_upper, '(\\d{5})(?:-\\d{4})?$', 1) <> ''
        THEN TRIM(REGEXP_REPLACE(street_no_unit, '\\s+', ' ', 'g')) || '|' ||
             REGEXP_EXTRACT(addr_upper, '(\\d{5})(?:-\\d{4})?$', 1)
        ELSE NULL
    END AS probate_addr_zip_key
FROM step2
""")

In [ ]:
#Match filtered probate addresses to property anchor
con.execute("""
CREATE OR REPLACE TABLE probate_property_match_filtered AS
WITH zip_match AS (
    SELECT
        p.*,
        a.acct,
        a.site_addr_1,
        a.situs_full,
        a.owner_entity_flag,
        a.absentee_owner_flag,
        'addr_zip_key' AS match_method
    FROM probate_party_addresses_keyed p
    INNER JOIN property_anchor a
        ON p.probate_addr_zip_key IS NOT NULL
       AND p.probate_addr_zip_key = a.addr_zip_key
),
no_unit_match AS (
    SELECT
        p.*,
        a.acct,
        a.site_addr_1,
        a.situs_full,
        a.owner_entity_flag,
        a.absentee_owner_flag,
        'addr_key_no_unit' AS match_method
    FROM probate_party_addresses_keyed p
    INNER JOIN property_anchor a
        ON p.probate_addr_key_no_unit IS NOT NULL
       AND p.probate_addr_key_no_unit = a.addr_key_no_unit
)
SELECT DISTINCT *
FROM zip_match
UNION
SELECT DISTINCT *
FROM no_unit_match
""")

In [ ]:
#Check match quality after filtering
con.execute("""
SELECT
    match_method,
    COUNT(*) AS rows_matched,
    COUNT(DISTINCT case_number) AS probate_cases_matched,
    COUNT(DISTINCT acct) AS properties_matched
FROM probate_property_match_filtered
GROUP BY 1
ORDER BY 2 DESC
""").df()

,match_method,rows_matched,probate_cases_matched,properties_matched
0,addr_key_no_unit,2056,713,788
1,addr_zip_key,1425,319,723


In [ ]:
#Inspect examples - these should look much more residential now
con.execute("""
SELECT
    case_number,
    case_file_date,
    party_role,
    party_name,
    raw_address,
    acct,
    site_addr_1,
    situs_full,
    match_method
FROM probate_property_match_filtered
LIMIT 50
""").df()

,case_number,case_file_date,party_role,party_name,raw_address,acct,site_addr_1,situs_full,match_method
0,489086,2020-10-19,Deceased,"Brooks, Gregory Dalton","6711 Seaton Valley Drive Spring, TX 77379",0332040140018,6711 STEARNS,6711 STEARNS HOUSTON 77021,addr_key_no_unit
1,445888,2016-01-25,Applicant,"FRANCIS, DEBRA KAY","6711 SPRING LEAF DR SPRING, TX 77379",0332040140018,6711 STEARNS,6711 STEARNS HOUSTON 77021,addr_key_no_unit
2,513100,2023-02-01,Independent Executor,"Thomas, Brittany Joann","9919 North P Street La Porte, TX 77571",0600520000144,9919 STEELMAN,9919 STEELMAN HOUSTON 77017,addr_key_no_unit
3,507441,2022-07-06,Applicant,"Beals, Janeen","8130 Sedona Ridge Drive Cypress, TX 77433",0790550060095,8130 STERLINGSHIRE,8130 STERLINGSHIRE HOUSTON 77078,addr_key_no_unit
4,507920,2022-07-21,Applicant,"Larry, Tiffany N.","8114 Amurwood Dr. Tomball, TX 77375",0790550060091,8114 STERLINGSHIRE,8114 STERLINGSHIRE HOUSTON 77078,addr_key_no_unit
5,450372,2016-07-15,Independent Administrator,"WEISINGER, PAUL","1405 SHADOWDALE DR APT#18 HOUSTON, TX 77043",1102750000005,1405 SHADOWDALE DR,1405 SHADOWDALE DR HOUSTON 77043,addr_key_no_unit
6,484386,2020-04-23,Applicant,"Fernandez, Debra M.","2379 Briarwest Blvd., Unit 13877 Houston, TX 7...",1075090010083,2379 BRIARWEST BLVD,2379 BRIARWEST BLVD HOUSTON 77077,addr_key_no_unit
7,531746,2025-01-07,Independent Executor,"Lyons, Brandi Ann","2100 Tanglewilde St., #371 Houston, TX 77063",1141580590003,2100 TANGLEWILDE ST 371,2100 TANGLEWILDE ST 371 HOUSTON 77063,addr_key_no_unit
8,448134,2016-04-15,Applicant,"HEIMBINDER, SHEILA M.","1111 HERMANN DR, UNIT 27 A HOUSTON, TX 77004",1155850250005,1111 HERMANN DR,1111 HERMANN DR HOUSTON 77004,addr_key_no_unit
9,535936,2025-06-13,Applicant,"Williams, Robert L., Jr.","914 Main St. Unit 1010 Houston,, TX 77002",0011380000010,914 MAIN ST,914 MAIN ST HOUSTON 77002,addr_key_no_unit


In [50]:
con.execute("""
SELECT
    party_role,
    COUNT(*) AS rows
FROM probate_parties_clean
GROUP BY 1
ORDER BY rows DESC
""").df()

,party_role,rows
0,Applicant,183291
1,Deceased,98263
2,Independent Executor,53947
3,Attorney Ad Litem (Participant),29363
4,Independent Executrix,13614
...,...,...
121,Temporary Guardian,2
122,Co-Administratrix,2
123,Temporary Administratrix Pending Contest,1
124,Converted Participant,1


In [51]:
con.execute("""
SELECT
    party_role,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN party_addr IS NOT NULL AND TRIM(party_addr) <> '' THEN 1 ELSE 0 END) AS rows_with_addr,
    ROUND(
        100.0 * SUM(CASE WHEN party_addr IS NOT NULL AND TRIM(party_addr) <> '' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_with_addr
FROM probate_parties_clean
GROUP BY 1
ORDER BY total_rows DESC
""").df()

,party_role,total_rows,rows_with_addr,pct_with_addr
0,Applicant,183291,169412.0,92.43
1,Deceased,98263,18636.0,18.97
2,Independent Executor,53947,50085.0,92.84
3,Attorney Ad Litem (Participant),29363,21637.0,73.69
4,Independent Executrix,13614,12197.0,89.59
...,...,...,...,...
121,Co-Successor Administrator with Will Annexed,2,2.0,100.00
122,Co-Administratrix,2,2.0,100.00
123,Temporary Administratrix Pending Contest,1,1.0,100.00
124,Converted Participant,1,1.0,100.00


In [52]:
con.execute("""
SELECT
    party_role,
    party_name,
    party_addr,
    party_addr2
FROM probate_parties_clean
WHERE party_addr IS NOT NULL
  AND TRIM(party_addr) <> ''
ORDER BY party_role, party_name
LIMIT 200
""").df()

,party_role,party_name,party_addr,party_addr2
0,Administrative,"Beck, Roger",5718 Westheimer Road Suite 1750,"Houston, TX 77057"
1,Administrative,"Campbell, Brandi",11767 Katy Freeway Suite 990,"Houston, TX 77079"
2,Administrative,"Chapa, Nilda",4423 Italy Ln,"Pasadena, TX 77505"
3,Administrative,"Goldberg, Michele",6750 West Loop South Suite 615,"Bellaire, TX 77401"
4,Administrative,"Hauser, Vanessa L","5300 Memorial #800 Houston, Tx 77007-8248",None
...,...,...,...,...
195,Administrator,"Ballard, Jacques Demetrius",1523 Pebble Chase Drive,"Katy, TX 77450"
196,Administrator,"Banks, Kevin Jerome",3409 Arbor Street,"Houston, TX 77004"
197,Administrator,"Banks, Kevin Jerome",3409 Arbor Street,"Houston, TX 77004"
198,Administrator,"Banks, Rachelle",6200 GREENS RD. APT. 2614,"HUMBLE, TX 77396"


In [ ]:
#I am focusing on decesased rows only now
con.execute("""
CREATE OR REPLACE TABLE probate_deceased_only AS
SELECT
    case_number,
    party_name AS deceased_name,
    party_addr,
    party_addr2
FROM probate_parties_clean
WHERE LOWER(party_role) LIKE '%deceased%'
""")

In [54]:
con.execute("""
SELECT
    COUNT(*) AS total_deceased,
    SUM(CASE WHEN party_addr IS NOT NULL AND TRIM(party_addr) <> '' THEN 1 ELSE 0 END) AS with_address
FROM probate_deceased_only
""").df()

,total_deceased,with_address
0,98263,18636.0


In [55]:
#filtering out bad addresses associated with a professional office under deceased
con.execute("""
CREATE OR REPLACE TABLE probate_addresses_filtered AS
SELECT *
FROM probate_parties_clean
WHERE party_addr IS NOT NULL
  AND TRIM(party_addr) <> ''

  -- remove obvious non-property addresses
  AND LOWER(party_addr) NOT LIKE '%suite%'
  AND LOWER(party_addr) NOT LIKE '%ste%'
  AND LOWER(party_addr) NOT LIKE '%floor%'
  AND LOWER(party_addr) NOT LIKE '%law%'
  AND LOWER(party_addr) NOT LIKE '%attorney%'
  AND LOWER(party_addr) NOT LIKE '%pllc%'
  AND LOWER(party_addr) NOT LIKE '%llp%'
""")

In [56]:
con.execute("""
SELECT
    p.case_number,
    p.party_role,
    p.party_name,
    p.party_addr
FROM probate_parties_clean p
WHERE p.case_number IN (
    SELECT case_number
    FROM probate_parties_clean
    WHERE LOWER(party_name) LIKE '%givans%'
)
ORDER BY case_number, party_role
""").df()

,case_number,party_role,party_name,party_addr
0,455884,Court Investigator,"Zinn, Tara",201 Caroline St. Suite 7000
1,455884,Court Visitor,"Zinn, Tara",201 Caroline St. Suite 7000
2,455884,Incapacitated,"GIVANS, MOURDY",811 GARNER RD
3,519699,Applicant,"Cooke, Edward Fenton, III",3141 Holy Road
4,519699,Deceased,"Givans, Patricia Ann",3141 Holy Road
5,519699,Independent Executor,"Cooke, Edward Fenton, III",3141 Holy Road


In [57]:
con.execute("""
CREATE OR REPLACE TABLE probate_case_address_groups AS
SELECT
    case_number,
    party_addr,

    COUNT(*) AS parties_at_address,

    SUM(CASE WHEN LOWER(party_role) LIKE '%deceased%' THEN 1 ELSE 0 END) AS deceased_count,

    STRING_AGG(party_role, ' | ') AS roles,
    STRING_AGG(party_name, ' | ') AS names

FROM probate_parties_clean
WHERE party_addr IS NOT NULL
  AND TRIM(party_addr) <> ''

GROUP BY case_number, party_addr
""")

In [58]:
con.execute("""
CREATE OR REPLACE TABLE probate_property_candidates AS
SELECT *
FROM probate_case_address_groups
WHERE
    deceased_count > 0          -- must include deceased
    AND parties_at_address >= 2 -- shared address (strong signal)
""")

In [59]:
con.execute("""
CREATE OR REPLACE TABLE probate_property_candidates_clean AS
SELECT *
FROM probate_property_candidates
WHERE LOWER(party_addr) NOT LIKE '%suite%'
  AND LOWER(party_addr) NOT LIKE '%ste%'
  AND LOWER(party_addr) NOT LIKE '%floor%'
  AND LOWER(party_addr) NOT LIKE '%law%'
  AND LOWER(party_addr) NOT LIKE '%llp%'
  AND LOWER(party_addr) NOT LIKE '%pllc%'
""")

In [60]:
con.execute("""
CREATE OR REPLACE TABLE probate_property_candidates_keys AS
SELECT
    *,
    UPPER(TRIM(party_addr)) AS addr_key_no_unit
FROM probate_property_candidates_clean
""")

In [61]:
con.execute("""
CREATE OR REPLACE TABLE probate_property_matches AS
SELECT
    p.case_number,
    p.party_addr,
    a.acct,
    a.site_addr_1,
    a.situs_full
FROM probate_property_candidates_keys p
JOIN property_anchor a
    ON UPPER(TRIM(a.site_addr_1)) = p.addr_key_no_unit
""")

In [62]:
con.execute("""
SELECT *
FROM
probate_property_matches""").df()

,case_number,party_addr,acct,site_addr_1,situs_full
0,460027,13902 Inland Spring ct,1172600160033,13902 INLAND SPRING CT,13902 INLAND SPRING CT HOUSTON 77059
1,462865,9615 Berlin Ct,1172650310013,9615 BERLIN CT,9615 BERLIN CT HOUSTON 77070
2,538365,19615 Southern Maple LN,1174030010029,19615 SOUTHERN MAPLE LN,19615 SOUTHERN MAPLE LN HOUSTON 77094
3,517068,11027 Wortham Blvd,1174160040027,11027 WORTHAM BLVD,11027 WORTHAM BLVD HOUSTON 77065
4,527782,8706 Bexar Dr,1174220010024,8706 BEXAR DR,8706 BEXAR DR HOUSTON 77064
...,...,...,...,...,...
481,518019,12821 High Star Dr,1034550010090,12821 HIGH STAR DR,12821 HIGH STAR DR HOUSTON 77072
482,472372,25110 Burnaby St,1035070000043,25110 BURNABY ST,25110 BURNABY ST SPRING 77373
483,519465,6011 Previn CT,1040250000019,6011 PREVIN CT,6011 PREVIN CT HOUSTON 77088
484,508538,10914 Plainfield ST,1040300000012,10914 PLAINFIELD ST,10914 PLAINFIELD ST HOUSTON 77031


In [67]:
con.execute("""
CREATE OR REPLACE TABLE probate_enriched_features AS
WITH base AS (
    SELECT *
    FROM gold.model_property_features_v5
),
probate_events_strict AS (
    SELECT
        acct,
        case_number,
        case_file_date AS probate_file_date
    FROM probate_property_match_filtered
),
probate_asof_snapshot AS (
    SELECT
        b.acct,
        b.snapshot_date,
        COUNT(DISTINCT p.case_number) AS probate_event_count_strict,
        MIN(p.probate_file_date) AS first_probate_date_strict,
        MAX(p.probate_file_date) AS last_probate_date_strict
    FROM base b
    LEFT JOIN probate_events_strict p
      ON b.acct = p.acct
     AND p.probate_file_date <= b.snapshot_date
    GROUP BY b.acct, b.snapshot_date
)
SELECT
    b.*,
    COALESCE(p.probate_event_count_strict, 0) AS probate_event_count_strict,
    CASE WHEN COALESCE(p.probate_event_count_strict, 0) > 0 THEN 1 ELSE 0 END AS has_probate_strict,
    p.first_probate_date_strict,
    p.last_probate_date_strict
FROM base b
LEFT JOIN probate_asof_snapshot p
  ON b.acct = p.acct
 AND b.snapshot_date = p.snapshot_date
""")

In [ ]:
#Keep only deceased rows for matching
con.execute("""
CREATE OR REPLACE TABLE probate_deceased_addresses AS
SELECT
    case_number,
    case_file_date,
    nature_of_proceeding,
    status,
    party_role,
    party_name,
    raw_address,
    UPPER(TRIM(raw_address)) AS addr_upper
FROM probate_party_addresses_filtered
WHERE LOWER(party_role) LIKE '%deceased%'
  AND raw_address IS NOT NULL
  AND TRIM(raw_address) <> ''
""")

In [ ]:
#Build a STRICT address key that keeps house number + street name
con.execute("""
CREATE OR REPLACE TABLE probate_deceased_addresses_keyed AS
WITH base AS (
    SELECT
        *,
        REGEXP_REPLACE(addr_upper, '[\\.,#]', '', 'g') AS addr_nopunct
    FROM probate_deceased_addresses
),
zip_extract AS (
    SELECT
        *,
        REGEXP_EXTRACT(addr_nopunct, '(\\d{5})(?:-\\d{4})?$', 1) AS probate_zip
    FROM base
),
state_strip AS (
    SELECT
        *,
        REGEXP_REPLACE(addr_nopunct, '\\s+TX\\s+\\d{5}(?:-\\d{4})?$', '', 'g') AS addr_no_state_zip
    FROM zip_extract
),
city_strip AS (
    SELECT
        *,
        REGEXP_REPLACE(
            addr_no_state_zip,
            '\\s+(HOUSTON|SPRING|CROSBY|KATY|TOMBALL|CYPRESS|PASADENA|LA PORTE|WEBSTER|KINGWOOD|HIGHLANDS|HUMBLE|CHANNELVIEW|BAYTOWN|SEABROOK|DEER PARK|SOUTH HOUSTON)$',
            '',
            'g'
        ) AS street_part
    FROM state_strip
),
unit_strip AS (
    SELECT
        *,
        REGEXP_REPLACE(
            street_part,
            '\\s+(APT|UNIT|STE|SUITE|LOT)\\s*[A-Z0-9-]+$',
            '',
            'g'
        ) AS street_no_unit
    FROM city_strip
)
SELECT
    case_number,
    case_file_date,
    nature_of_proceeding,
    status,
    party_role,
    party_name,
    raw_address,
    TRIM(REGEXP_REPLACE(street_no_unit, '\\s+', ' ', 'g')) AS probate_addr_key_no_unit,
    probate_zip,
    CASE
        WHEN probate_zip <> ''
        THEN TRIM(REGEXP_REPLACE(street_no_unit, '\\s+', ' ', 'g')) || '|' || probate_zip
        ELSE NULL
    END AS probate_addr_zip_key
FROM unit_strip
WHERE TRIM(REGEXP_REPLACE(street_no_unit, '\\s+', ' ', 'g')) <> ''
""")

In [ ]:
#Inspect the new keys - they should look like full streets, not just numbers
con.execute("""
SELECT
    raw_address,
    probate_addr_key_no_unit,
    probate_addr_zip_key
FROM probate_deceased_addresses_keyed
LIMIT 50
""").df()

,raw_address,probate_addr_key_no_unit,probate_addr_zip_key
0,"11814 Miramar Shores Drive Houston, TX 77065",11814 MIRAMAR SHORES DRIVE,11814 MIRAMAR SHORES DRIVE|77065
1,"4914 Libbey Lane Houston, TX 77092",4914 LIBBEY LANE,4914 LIBBEY LANE|77092
2,"4404 Blossom St. Houston, TX 77007",4404 BLOSSOM ST,4404 BLOSSOM ST|77007
3,"820 Artemis N/A Webster, TX 77598",820 ARTEMIS N/A,820 ARTEMIS N/A|77598
4,"10911 Sunnydale Ridge Lane Cypress, TX 77433",10911 SUNNYDALE RIDGE LANE,10911 SUNNYDALE RIDGE LANE|77433
5,"2214 Southgate Houston, TX 77030",2214 SOUTHGATE,2214 SOUTHGATE|77030
6,"1828 Bissonnet St. Houston, TX 77005",1828 BISSONNET ST,1828 BISSONNET ST|77005
7,"8727 Dawnblush Lane Houston, TX 77095",8727 DAWNBLUSH LANE,8727 DAWNBLUSH LANE|77095
8,"3207 Cypresswood Drive Spring, TX 77388",3207 CYPRESSWOOD DRIVE,3207 CYPRESSWOOD DRIVE|77388
9,"7818 Glenn Cliff Dr Houston, TX 77064",7818 GLENN CLIFF DR,7818 GLENN CLIFF DR|77064


In [ ]:
#Match deceased-only addresses to property anchor
con.execute("""
CREATE OR REPLACE TABLE probate_property_match_strict AS
WITH zip_match AS (
    SELECT
        p.*,
        a.acct,
        a.site_addr_1,
        a.situs_full,
        'addr_zip_key' AS match_method
    FROM probate_deceased_addresses_keyed p
    INNER JOIN property_anchor a
        ON p.probate_addr_zip_key IS NOT NULL
       AND p.probate_addr_zip_key = a.addr_zip_key
),
no_unit_match AS (
    SELECT
        p.*,
        a.acct,
        a.site_addr_1,
        a.situs_full,
        'addr_key_no_unit' AS match_method
    FROM probate_deceased_addresses_keyed p
    INNER JOIN property_anchor a
        ON p.probate_addr_key_no_unit = a.addr_key_no_unit
)
SELECT DISTINCT *
FROM zip_match
UNION
SELECT DISTINCT *
FROM no_unit_match
""")

In [ ]:
#Check match counts
con.execute("""
SELECT
    match_method,
    COUNT(*) AS rows_matched,
    COUNT(DISTINCT case_number) AS probate_cases_matched,
    COUNT(DISTINCT acct) AS properties_matched
FROM probate_property_match_strict
GROUP BY 1
ORDER BY 2 DESC
""").df()

,match_method,rows_matched,probate_cases_matched,properties_matched
0,addr_key_no_unit,3031,2932,2820
1,addr_zip_key,2928,2846,2729


In [ ]:
#Spot check examples
con.execute("""
SELECT
    case_number,
    case_file_date,
    party_role,
    party_name,
    raw_address,
    probate_addr_key_no_unit,
    probate_addr_zip_key,
    acct,
    site_addr_1,
    situs_full,
    match_method
FROM probate_property_match_strict
LIMIT 50
""").df()

,case_number,case_file_date,party_role,party_name,raw_address,probate_addr_key_no_unit,probate_addr_zip_key,acct,site_addr_1,situs_full,match_method
0,451760,2016-09-09,Deceased,"Williford, Allen H.","4511 Village Corner Dr Houston, TX 77059",4511 VILLAGE CORNER DR,4511 VILLAGE CORNER DR|77059,1173450080026,4511 VILLAGE CORNER DR,4511 VILLAGE CORNER DR HOUSTON 77059,addr_key_no_unit
1,511685,2022-12-07,Deceased,"McGINTY, PEGGY JOAN","4102 DANEK RD CROSBY, TX 77532",4102 DANEK RD,4102 DANEK RD|77532,0591390000208,4102 DANEK RD,4102 DANEK RD CROSBY 77532,addr_key_no_unit
2,465428,2018-03-20,Deceased,"O'Connor, Michael","224 Birdsall St Houston, TX 77007",224 BIRDSALL ST,224 BIRDSALL ST|77007,0300690000046,224 BIRDSALL ST,224 BIRDSALL ST HOUSTON 77007,addr_key_no_unit
3,490806,2020-12-28,Deceased,"Bedford, Caleb","4822 Mallow St. Houston, TX 77033",4822 MALLOW ST,4822 MALLOW ST|77033,0751990090006,4822 MALLOW ST,4822 MALLOW ST HOUSTON 77033,addr_key_no_unit
4,472111,2018-11-30,Deceased,"Lopez, Carmen Quiroz","12821 Northumb Rd. Houston, TX 77047",12821 NORTHUMB RD,12821 NORTHUMB RD|77047,0842020000009,12821 NORTHUMB RD,12821 NORTHUMB RD HOUSTON 77047,addr_key_no_unit
5,454445,2017-01-09,Deceased,"Chapman, Fredda Scroggins","11519 Atwell Dr. Houston, TX 77035",11519 ATWELL DR,11519 ATWELL DR|77035,0860140000002,11519 ATWELL DR,11519 ATWELL DR HOUSTON 77035,addr_key_no_unit
6,471503,2018-11-05,Deceased,"Sefton, Bonnie Leon","1804 Marlock Ln. Pasadena, TX 77502",1804 MARLOCK LN,1804 MARLOCK LN|77502,0862010000013,1804 MARLOCK LN,1804 MARLOCK LN PASADENA 77502,addr_key_no_unit
7,460312,2017-08-21,Deceased,"Welch, Neil H., Sr.","11723 Possum Hollow Ln. Houston, TX 77065",11723 POSSUM HOLLOW LN,11723 POSSUM HOLLOW LN|77065,0903640000083,11723 POSSUM HOLLOW LN,11723 POSSUM HOLLOW LN HOUSTON 77065,addr_key_no_unit
8,514991,2023-04-05,Deceased,"Jacobs, Arturo Alfonso, Sr.","12302 Laneview Dr. Houston, TX 77070",12302 LANEVIEW DR,12302 LANEVIEW DR|77070,1142700240011,12302 LANEVIEW DR,12302 LANEVIEW DR HOUSTON 77070,addr_key_no_unit
9,514229,2023-03-09,Deceased,"Garcia, Consuelo Ojeda","12707 Summer Mill Dr. Houston, TX 77070",12707 SUMMER MILL DR,12707 SUMMER MILL DR|77070,1145530170021,12707 SUMMER MILL DR,12707 SUMMER MILL DR HOUSTON 77070,addr_key_no_unit


In [ ]:
#adding probate events back into the ML table
con.execute("""
CREATE OR REPLACE TABLE gold.model_property_features_v8 AS
WITH base AS (
    SELECT *
    FROM gold.model_property_features_v5
),
probate_events_strict AS (
    SELECT DISTINCT
        acct,
        case_number,
        case_file_date AS probate_file_date
    FROM probate_property_match_strict
),
probate_asof_snapshot AS (
    SELECT
        b.acct,
        b.snapshot_date,
        COUNT(DISTINCT p.case_number) AS probate_event_count_strict,
        MIN(p.probate_file_date) AS first_probate_date_strict,
        MAX(p.probate_file_date) AS last_probate_date_strict
    FROM base b
    LEFT JOIN probate_events_strict p
      ON b.acct = p.acct
     AND p.probate_file_date <= b.snapshot_date
    GROUP BY b.acct, b.snapshot_date
)
SELECT
    b.*,
    COALESCE(p.probate_event_count_strict, 0) AS probate_event_count_strict,
    CASE WHEN COALESCE(p.probate_event_count_strict, 0) > 0 THEN 1 ELSE 0 END AS has_probate_strict,
    p.first_probate_date_strict,
    p.last_probate_date_strict,
    CASE
        WHEN p.last_probate_date_strict IS NOT NULL
        THEN DATE_DIFF('day', p.last_probate_date_strict, b.snapshot_date)
        ELSE NULL
    END AS days_since_probate_strict,
    CASE
        WHEN p.last_probate_date_strict IS NOT NULL
         AND DATE_DIFF('day', p.last_probate_date_strict, b.snapshot_date) BETWEEN 0 AND 365
        THEN 1 ELSE 0
    END AS probate_strict_last_12mo,
    CASE
        WHEN p.last_probate_date_strict IS NOT NULL
         AND DATE_DIFF('day', p.last_probate_date_strict, b.snapshot_date) BETWEEN 0 AND 730
        THEN 1 ELSE 0
    END AS probate_strict_last_24mo,
    CASE
        WHEN p.last_probate_date_strict IS NOT NULL
         AND DATE_DIFF('day', p.last_probate_date_strict, b.snapshot_date) BETWEEN 0 AND 913
        THEN 1 ELSE 0
    END AS probate_strict_last_30mo,
    CASE
        WHEN p.last_probate_date_strict IS NOT NULL
         AND DATE_DIFF('day', p.last_probate_date_strict, b.snapshot_date) BETWEEN 0 AND 1095
        THEN 1 ELSE 0
    END AS probate_strict_last_36mo,
    CASE
        WHEN p.last_probate_date_strict IS NOT NULL
         AND DATE_DIFF('day', p.last_probate_date_strict, b.snapshot_date) BETWEEN 0 AND 1217
        THEN 1 ELSE 0
    END AS probate_strict_last_40mo
FROM base b
LEFT JOIN probate_asof_snapshot p
  ON b.acct = p.acct
 AND b.snapshot_date = p.snapshot_date
""")

In [77]:
con.execute("""
SELECT
    has_probate_strict,
    COUNT(*) AS properties,
    SUM(sold_next_12mo) AS sales,
    ROUND(100.0 * AVG(sold_next_12mo), 2) AS sale_rate_pct
FROM gold.model_property_features_v8
GROUP BY 1
ORDER BY 1
""").df()

,has_probate_strict,properties,sales,sale_rate_pct
0,0,256278,23810.0,9.29
1,1,765,76.0,9.93


In [78]:
con.execute("""
SELECT
    probate_strict_last_24mo,
    COUNT(*) AS properties,
    SUM(sold_next_12mo) AS sales,
    ROUND(100.0 * AVG(sold_next_12mo), 2) AS sale_rate_pct
FROM gold.model_property_features_v8
GROUP BY 1
ORDER BY 1
""").df()

,probate_strict_last_24mo,properties,sales,sale_rate_pct
0,0,256799,23848.0,9.29
1,1,244,38.0,15.57
